# Quick test to see how TabPFN performs on this data

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from tabpfn import TabPFNClassifier
from utils import score_with_thresh
from sklearn.ensemble import RandomForestClassifier

SEED = 3105
rng = np.random.default_rng(SEED)
np.random.seed(SEED)

# numbers of features to test
K_VALUES = [2, 3, 4]
N_RANDOM = 3

/Users/ola/projects/cost-sensitive-marketing/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
data_dir = Path("../../data")
X = pd.read_csv(data_dir / "x_train.txt", sep=" ")
y = pd.read_csv(data_dir / "y_train.txt", sep=" ").values.ravel()

with open("../feature_selection/selected_features.txt") as f:
    selected = [s.strip().strip("'").strip('"') for s in f.read().split(",")]

X = X[selected]
print(f"X: {X.shape}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

X: (5000, 30)


In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)
ranking = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
top15 = ranking.index[:15].tolist()

def random_subsets(k, n, excludes):
    seen = {frozenset(e) for e in excludes}
    subsets = []
    while len(subsets) < n:
        s = frozenset(rng.choice(top15, size=k, replace=False))
        if s not in seen:
            seen.add(s)
            subsets.append(sorted(s, key=top15.index))
    return subsets


subsets = {}
for k in K_VALUES:
    topk = ranking.index[:k].tolist()
    topk_1 = ranking.index[1 : k + 1].tolist()
    topk_2 = ranking.index[2 : k + 2].tolist()
    subsets[k] = (
        [("top", topk)]
        + [("top+1", topk_1)]
        + [("top+2", topk_2)]
        + [(f"rand_{i}", s) for i, s in enumerate(random_subsets(k, N_RANDOM, [topk, topk_1,topk_2]))]
    )
    print(f"k={k}: {len(subsets[k])} subsets")

k=2: 6 subsets
k=3: 6 subsets
k=4: 6 subsets


In [15]:
best_thresholds = []
best_scores = []

for k in K_VALUES:
    print(f"k={k}")
    for subset_id, (kind, features) in enumerate(subsets[k]):
        X_train_tmp = X_train[features]

        clf = TabPFNClassifier(
            device="cpu", 
            ignore_pretraining_limits=True,
            n_estimators=10, random_state=3105
        )
        clf.fit(X_train_tmp, y_train)
        
        proba = clf.predict_proba(X_test[features])[:,1]
        score, contacted = score_with_thresh(y_test, proba, n_var=k, thresh=0)
        print(f"    subset {kind}, score={score}, contacted={contacted}")

k=2
    subset top, score=535, contacted=200
    subset top+1, score=520, contacted=200
    subset top+2, score=595, contacted=200
    subset rand_0, score=535, contacted=200
    subset rand_1, score=550, contacted=200
    subset rand_2, score=460, contacted=200
k=3
    subset top, score=560, contacted=200
    subset top+1, score=515, contacted=200
    subset top+2, score=560, contacted=200
    subset rand_0, score=560, contacted=200
    subset rand_1, score=260, contacted=200
    subset rand_2, score=320, contacted=200
k=4
    subset top, score=450, contacted=200
    subset top+1, score=300, contacted=200
    subset top+2, score=375, contacted=200
    subset rand_0, score=165, contacted=200
    subset rand_1, score=180, contacted=200
    subset rand_2, score=30, contacted=200
